In [10]:
import numpy as np
import pandas as pd
import os, math, glob, random
import xarray as xr
from tqdm.notebook import tqdm  
import shutil
import time as pytime

In [11]:
# ========= 全局参数区 =========
# URL 根据变量修改:
# 温度 T: t3z | 盐度 S: s3z | 流速 U: u3z | 流速 V: v3z
URL = "https://tds.hycom.org/thredds/dodsC/ESPC-D-V02/u3z/2025" 
VAR_NAME = "water_u"  # 可改为 "salinity", "water_u", "water_v", "water_temp"
OUT_ROOT = r"C:\Project\2025_HT\Input\HYCOM_2025_monthly"

# 空间裁剪范围
LON_RANGE = (116.0, 123.0)
LAT_RANGE = (20.5, 27.5)
DEPTH_RANGE = (0.0, 3800.0)

os.makedirs(OUT_ROOT, exist_ok=True)

In [12]:
month_time_index = {
    1:  (0, 247),
    2:  (248, 471),
    3:  (472, 719),
    4:  (720, 959),
    5:  (960, 1207),
    6:  (1208, 1447),
    7:  (1448, 1695),
    8:  (1696, 1943),
    9:  (1944, 2183),
    10: (2184, 2431),
    11: (2432, 2671),
    12: (2672, 2919),
} # 注意闰年替换

In [13]:
def range_to_index(axis_1d, vmin, vmax, outside=True):
    """(保留你原有的空间映射逻辑) 将物理范围映射到索引"""
    axis = np.asarray(axis_1d, dtype=float)
    vmin, vmax = float(vmin), float(vmax)
    reversed_axis = axis[0] > axis[-1]
    axis_work = axis[::-1] if reversed_axis else axis
    lo, hi = (vmin, vmax) if vmin <= vmax else (vmax, vmin)

    i0, i1 = int(np.abs(axis_work - lo).argmin()), int(np.abs(axis_work - hi).argmin())
    if i0 > i1: i0, i1 = i1, i0
    if outside:
        if axis_work[i0] > lo and i0 > 0: i0 -= 1
        if axis_work[i1] < hi and i1 < len(axis_work) - 1: i1 += 1
    if reversed_axis:
        n = len(axis)
        i0, i1 = n - 1 - i1, n - 1 - i0
    return i0, i1, float(axis[i0]), float(axis[i1])

# 获取空间索引
ds_temp = xr.open_dataset(URL, decode_times=False)
lon, lat, depth = ds_temp["lon"].values, ds_temp["lat"].values, ds_temp["depth"].values
ds_temp.close()

i0, i1, _, _ = range_to_index(lon, *LON_RANGE)
j0, j1, _, _ = range_to_index(lat, *LAT_RANGE)
k0, k1, _, _ = range_to_index(depth, *DEPTH_RANGE)

print(f"空间索引 - lon:{i0}-{i1}, lat:{j0}-{j1}, depth:{k0}-{k1}")

空间索引 - lon:1450-1538, lat:2512-2688, depth:0-38


In [15]:
# ========= 工具函数 =========
def atomic_write_netcdf(obj, out_file, engine="netcdf4"):
    tmp = out_file + ".tmp"
    os.makedirs(os.path.dirname(out_file), exist_ok=True)
    try:
        if os.path.exists(tmp):
            os.remove(tmp)
        obj.to_netcdf(tmp, engine=engine)
        os.replace(tmp, out_file)
    except Exception:
        # 清理残留 tmp，避免下次干扰
        if os.path.exists(tmp):
            try:
                os.remove(tmp)
            except Exception:
                pass
        raise

def qc_raw_zero_frac(nc_file, zero_thresh=0.2):
    """
    用 decode_cf=False 读取 raw（不解码scale/offset），检查 raw==0 污染比例。
    只检查 depth=0（表层）即可高敏感抓到你这种问题。
    """
    d = xr.open_dataset(nc_file, decode_cf=False, decode_times=False)
    
    # 同时兼容 water_temp / salinity / water_u / water_v，优先按这个顺序
    for candidate in ("water_temp", "salinity", "water_u", "water_v"):
        if candidate in d.data_vars:
            var_name = candidate
            break
    else:
        # 回退到第一个数据变量（防护）
        var_name = list(d.data_vars)[0]
        
    raw = d[var_name].isel(depth=0)   # (time, lat, lon) raw int16

    a = raw.values
    a = a[np.isfinite(a)]
    if a.size == 0:
        d.close()
        return False, 1.0
    zero_frac = float(np.mean(a == 0))
    d.close()
    return (zero_frac <= zero_thresh), zero_frac

def part_len_ok(nc_file, expected_time_len):
    if not os.path.exists(nc_file):
        return False
    try:
        with xr.open_dataset(nc_file, decode_times=False) as d:
            return ("time" in d.sizes) and (d.sizes["time"] == expected_time_len)
    except Exception:
        return False

def download_part_with_qc(sub_ds, part_file, expected_time_len,
                          zero_thresh=0.2, max_retries=5):
    for attempt in range(max_retries):
        try:
            atomic_write_netcdf(sub_ds, part_file)
        except Exception as e:
            # 这里把 “NetCDF: file not found”等异常也当作一次失败进入重试
            backoff_sleep(attempt)
            continue

        if not part_len_ok(part_file, expected_time_len):
            if os.path.exists(part_file):
                os.remove(part_file)
            backoff_sleep(attempt)
            continue

        ok, zf = qc_raw_zero_frac(part_file, zero_thresh=zero_thresh)
        if ok:
            return True, zf

        if os.path.exists(part_file):
            os.remove(part_file)
        backoff_sleep(attempt)

    return False, None

def month_is_complete(month_file, expected_time_len):
    return part_len_ok(month_file, expected_time_len)


def backoff_sleep(attempt):
    pytime.sleep(min(600, 2**attempt) + random.random()*2)


In [16]:
url = "https://tds.hycom.org/thredds/dodsC/ESPC-D-V02/u3z/2025"
out_root = r"C:\Project\2025_HT\Input\HYCOM_2025_monthly_U_qc"
os.makedirs(out_root, exist_ok=True)
# 越小越稳（公司网慢/服务器抖动大：推荐 16 或 24）
chunk_time   = 4     # 24个time步=3天(3小时一次：8步/天)
zero_thresh  = 0.01   # raw==0 占比 > 20% 认为“半坏/坏”
max_retries  = 20      # 每块最多重试次数

In [17]:
# ========= 打开远端数据集 =========
ds = xr.open_dataset(url, decode_times=False)
if "tau" in ds:
    ds = ds.drop_vars("tau")

# ========= 主循环：按月分块下载 + QC =========
for month in range(1,13):
    t0, t1 = month_time_index[month]
    expected_month_len = t1 - t0 + 1

    month_file = os.path.join(out_root, f"HYCOM_2025_U_M{month:02d}.nc")
    parts_dir  = os.path.join(out_root, f"M{month:02d}_parts")
    os.makedirs(parts_dir, exist_ok=True)

    if month_is_complete(month_file, expected_month_len):
        print(f"M{month:02d} complete -> skip")
        continue

    n_chunks = math.ceil(expected_month_len / chunk_time)

    with tqdm(total=n_chunks, desc=f"Month {month:02d}", leave=True) as pbar:
        for k in range(n_chunks):
            s = t0 + k * chunk_time
            e = min(t1 + 1, s + chunk_time)
            part_len = e - s

            part_file = os.path.join(parts_dir, f"part_{k:03d}_{s}_{e-1}.nc")

            # 已存在则先QC，坏就删掉重下
            if part_len_ok(part_file, part_len):
                ok, zf = qc_raw_zero_frac(part_file, zero_thresh=zero_thresh)
                if ok:
                    pbar.update(1)
                    pbar.set_postfix({"cached_zf": f"{zf:.3f}"})
                    continue
                else:
                    os.remove(part_file)

            # 取子集：只取 water_u（减少传输）
            sub = ds[["water_u"]].isel(
                time=slice(s, e),
                depth=slice(k0, k1 + 1),
                lat=slice(j0, j1 + 1),
                lon=slice(i0, i1 + 1)
            )

            success, zf = download_part_with_qc(
                sub, part_file, part_len,
                zero_thresh=zero_thresh, max_retries=max_retries
            )

            if not success:
                print(f"[Warn] M{month:02d} chunk {k} failed after retries. Re-run will resume.")
                break

            pbar.update(1)
            pbar.set_postfix({"zf": f"{zf:.3f}"})

    # part齐了才合并
    part_files = sorted(glob.glob(os.path.join(parts_dir, "part_*.nc")))
    if len(part_files) != n_chunks:
        continue

    ds_month = xr.open_mfdataset(
        part_files, combine="nested", concat_dim="time", decode_times=False
    )

    atomic_write_netcdf(ds_month, month_file)
    print(f"Saved: {month_file}")


Month 01:   0%|          | 0/62 [00:00<?, ?it/s]

Saved: C:\Project\2025_HT\Input\HYCOM_2025_monthly_U_qc\HYCOM_2025_U_M01.nc


Month 02:   0%|          | 0/56 [00:00<?, ?it/s]

Saved: C:\Project\2025_HT\Input\HYCOM_2025_monthly_U_qc\HYCOM_2025_U_M02.nc


Month 03:   0%|          | 0/62 [00:00<?, ?it/s]

Saved: C:\Project\2025_HT\Input\HYCOM_2025_monthly_U_qc\HYCOM_2025_U_M03.nc


... (14 outputs left out)

Month 11:   0%|          | 0/60 [00:00<?, ?it/s]

Saved: C:\Project\2025_HT\Input\HYCOM_2025_monthly_U_qc\HYCOM_2025_U_M11.nc


Month 12:   0%|          | 0/62 [00:00<?, ?it/s]

Saved: C:\Project\2025_HT\Input\HYCOM_2025_monthly_U_qc\HYCOM_2025_U_M12.nc


In [18]:
url = "https://tds.hycom.org/thredds/dodsC/ESPC-D-V02/v3z/2025"
out_root = r"C:\Project\2025_HT\Input\HYCOM_2025_monthly_V_qc"
os.makedirs(out_root, exist_ok=True)
# 越小越稳（公司网慢/服务器抖动大：推荐 16 或 24）
chunk_time   = 4     # 24个time步=3天(3小时一次：8步/天)
zero_thresh  = 0.01   # raw==0 占比 > 20% 认为“半坏/坏”
max_retries  = 20      # 每块最多重试次数

In [19]:
# ========= 打开远端数据集 =========
ds = xr.open_dataset(url, decode_times=False)
if "tau" in ds:
    ds = ds.drop_vars("tau")

# ========= 主循环：按月分块下载 + QC =========
for month in range(1,13):
    t0, t1 = month_time_index[month]
    expected_month_len = t1 - t0 + 1

    month_file = os.path.join(out_root, f"HYCOM_2025_V_M{month:02d}.nc")
    parts_dir  = os.path.join(out_root, f"M{month:02d}_parts")
    os.makedirs(parts_dir, exist_ok=True)

    if month_is_complete(month_file, expected_month_len):
        print(f"M{month:02d} complete -> skip")
        continue

    n_chunks = math.ceil(expected_month_len / chunk_time)

    with tqdm(total=n_chunks, desc=f"Month {month:02d}", leave=True) as pbar:
        for k in range(n_chunks):
            s = t0 + k * chunk_time
            e = min(t1 + 1, s + chunk_time)
            part_len = e - s

            part_file = os.path.join(parts_dir, f"part_{k:03d}_{s}_{e-1}.nc")

            # 已存在则先QC，坏就删掉重下
            if part_len_ok(part_file, part_len):
                ok, zf = qc_raw_zero_frac(part_file, zero_thresh=zero_thresh)
                if ok:
                    pbar.update(1)
                    pbar.set_postfix({"cached_zf": f"{zf:.3f}"})
                    continue
                else:
                    os.remove(part_file)

            # 取子集：只取 water_v（减少传输）
            sub = ds[["water_v"]].isel(
                time=slice(s, e),
                depth=slice(k0, k1 + 1),
                lat=slice(j0, j1 + 1),
                lon=slice(i0, i1 + 1)
            )

            success, zf = download_part_with_qc(
                sub, part_file, part_len,
                zero_thresh=zero_thresh, max_retries=max_retries
            )

            if not success:
                print(f"[Warn] M{month:02d} chunk {k} failed after retries. Re-run will resume.")
                break

            pbar.update(1)
            pbar.set_postfix({"zf": f"{zf:.3f}"})

    # part齐了才合并
    part_files = sorted(glob.glob(os.path.join(parts_dir, "part_*.nc")))
    if len(part_files) != n_chunks:
        continue

    ds_month = xr.open_mfdataset(
        part_files, combine="nested", concat_dim="time", decode_times=False
    )

    atomic_write_netcdf(ds_month, month_file)
    print(f"Saved: {month_file}")


Month 01:   0%|          | 0/62 [00:00<?, ?it/s]

Saved: C:\Project\2025_HT\Input\HYCOM_2025_monthly_V_qc\HYCOM_2025_V_M01.nc


Month 02:   0%|          | 0/56 [00:00<?, ?it/s]

Saved: C:\Project\2025_HT\Input\HYCOM_2025_monthly_V_qc\HYCOM_2025_V_M02.nc


Month 03:   0%|          | 0/62 [00:00<?, ?it/s]

Saved: C:\Project\2025_HT\Input\HYCOM_2025_monthly_V_qc\HYCOM_2025_V_M03.nc


... (14 outputs left out)

Month 11:   0%|          | 0/60 [00:00<?, ?it/s]

Saved: C:\Project\2025_HT\Input\HYCOM_2025_monthly_V_qc\HYCOM_2025_V_M11.nc


Month 12:   0%|          | 0/62 [00:00<?, ?it/s]

Saved: C:\Project\2025_HT\Input\HYCOM_2025_monthly_V_qc\HYCOM_2025_V_M12.nc


### single month repa

In [12]:
chunk_time   = 1     # 24个time步=3天(3小时一次：8步/天)
zero_thresh  = 0.01   # raw==0 占比 > 20% 认为“半坏/坏”
max_retries  = 4      # 每块最多重试次数
# =========================
SKIP_TIME_IDX = {2741}

# 失败也生成占位，保证能继续往后 & 能合并
MAKE_PLACEHOLDER_ON_FAIL = True

# 给占位文件加一个轻量标记文件，避免下次又去 QC/重下
def placeholder_marker(part_file: str) -> str:
    return part_file + ".PLACEHOLDER"

def is_placeholder(part_file: str) -> bool:
    return os.path.exists(placeholder_marker(part_file))

def touch_placeholder_marker(part_file: str, reason: str = ""):
    mk = placeholder_marker(part_file)
    with open(mk, "w", encoding="utf-8") as f:
        f.write(reason)

def make_nan_placeholder_from_ds(ds, part_file, s, e, k0, k1, j0, j1, i0, i1, varname="salinity"):
    """
    生成一个与真实 part 相同维度/坐标的 NaN 占位文件：
    dims: time, depth, lat, lon
    只读取很小的坐标变量（time/depth/lat/lon），不会去拉大数组。
    """
    os.makedirs(os.path.dirname(part_file), exist_ok=True)

    # 只取坐标（小）
    time_vals  = ds["time"].isel(time=slice(s, e)).values
    depth_vals = ds["depth"].isel(depth=slice(k0, k1 + 1)).values
    lat_vals   = ds["lat"].isel(lat=slice(j0, j1 + 1)).values
    lon_vals   = ds["lon"].isel(lon=slice(i0, i1 + 1)).values

    shape = (len(time_vals), len(depth_vals), len(lat_vals), len(lon_vals))
    data = np.full(shape, np.nan, dtype=np.float32)

    da = xr.DataArray(
        data,
        dims=("time", "depth", "lat", "lon"),
        coords={"time": time_vals, "depth": depth_vals, "lat": lat_vals, "lon": lon_vals},
        name=varname
    )
    ds_ph = da.to_dataset()

    # 写文件（可选加压缩；如果你环境支持 netcdf4，会更省磁盘）
    # encoding = {varname: {"zlib": True, "complevel": 1, "dtype": "float32"}}
    # ds_ph.to_netcdf(part_file, encoding=encoding)

    ds_ph.to_netcdf(part_file)
    touch_placeholder_marker(part_file, reason=f"placeholder: {varname}, time={s}..{e-1}\n")
    return True

In [13]:
# ========= 打开远端数据集 =========
ds = xr.open_dataset(url, decode_times=False)
if "tau" in ds:
    ds = ds.drop_vars("tau")

# ========= 主循环：按月分块下载 + QC + 占位 =========
for month in range(12, 13):
    t0, t1 = month_time_index[month]
    expected_month_len = t1 - t0 + 1

    month_file = os.path.join(out_root, f"HYCOM_2025_salinity_M{month:02d}.nc")
    parts_dir  = os.path.join(out_root, f"M{month:02d}_parts")
    os.makedirs(parts_dir, exist_ok=True)

    if month_is_complete(month_file, expected_month_len):
        print(f"M{month:02d} complete -> skip")
        continue

    n_chunks = math.ceil(expected_month_len / chunk_time)

    with tqdm(total=n_chunks, desc=f"Month {month:02d}", leave=True) as pbar:
        for k in range(n_chunks):
            s = t0 + k * chunk_time
            e = min(t1 + 1, s + chunk_time)
            part_len = e - s

            part_file = os.path.join(parts_dir, f"part_{k:03d}_{s}_{e-1}.nc")

            # -----------------------------
            # (A) 手动跳过：生成占位并继续
            # -----------------------------
            if chunk_time == 1 and s in SKIP_TIME_IDX:
                if not os.path.exists(part_file):
                    make_nan_placeholder_from_ds(ds, part_file, s, e, k0, k1, j0, j1, i0, i1, varname="salinity")
                print(f"[Skip->Placeholder] M{month:02d} time={s}")
                pbar.update(1)
                pbar.set_postfix({"skip": str(s)})
                continue

            # -----------------------------
            # (B) 已存在：占位则直接算完成；真实文件则 QC，不通过就删掉重下
            # -----------------------------
            if part_len_ok(part_file, part_len):
                if is_placeholder(part_file):
                    pbar.update(1)
                    pbar.set_postfix({"ph": f"{s}"})
                    continue

                ok, zf = qc_raw_zero_frac(part_file, zero_thresh=zero_thresh)
                if ok:
                    pbar.update(1)
                    pbar.set_postfix({"cached_zf": f"{zf:.3f}"})
                    continue
                else:
                    # 坏文件删掉（同时如果有旧 marker 也删）
                    try:
                        os.remove(part_file)
                    except FileNotFoundError:
                        pass
                    mk = placeholder_marker(part_file)
                    if os.path.exists(mk):
                        os.remove(mk)

            # -----------------------------
            # (C) 正常下载：只取 salinity 子集
            # -----------------------------
            sub = ds[["salinity"]].isel(
                time=slice(s, e),
                depth=slice(k0, k1 + 1),
                lat=slice(j0, j1 + 1),
                lon=slice(i0, i1 + 1)
            )

            success, zf = download_part_with_qc(
                sub, part_file, part_len,
                zero_thresh=zero_thresh, max_retries=max_retries
            )

            if not success:
                print(f"[Warn] M{month:02d} chunk {k} time={s} failed after retries.")

                # (D) 失败也生成占位：不中断循环，继续后面的 chunk
                if MAKE_PLACEHOLDER_ON_FAIL:
                    try:
                        make_nan_placeholder_from_ds(ds, part_file, s, e, k0, k1, j0, j1, i0, i1, varname="salinity")
                        print(f"[Fail->Placeholder] created for time={s}")
                    except Exception as ex:
                        print(f"[Fail->Placeholder] failed to create placeholder for time={s}: {ex}")
                        # 这里仍然继续，别卡死
                pbar.update(1)
                pbar.set_postfix({"failed": str(s)})
                continue

            # 成功
            pbar.update(1)
            pbar.set_postfix({"zf": f"{zf:.3f}"})

    # =========================
    # 合并：只要 part 数量齐就合并（占位也算齐）
    # =========================
    part_files = sorted(glob.glob(os.path.join(parts_dir, "part_*.nc")))
    if len(part_files) != n_chunks:
        print(f"[Info] M{month:02d} parts not complete ({len(part_files)}/{n_chunks}), skip merge.")
        continue

    ds_month = xr.open_mfdataset(
        part_files, combine="nested", concat_dim="time", decode_times=False
    )

    atomic_write_netcdf(ds_month, month_file)
    print(f"Saved: {month_file}")

Month 12:   0%|          | 0/248 [00:00<?, ?it/s]

[Skip->Placeholder] M12 time=2741
Saved: C:\Project\2025_HT\Input\HYCOM_2025_monthly_S_qc\HYCOM_2025_salinity_M12.nc
